# Memory-Native Qwen3.8-27B — Production Recovery 3k (v4)

**Compute-efficient production baseline 3k** for **RTX PRO 6000 Blackwell 96 GB** (H100 also fine).
Descends from the archived v3 recipe; governed by `production/qwen38_27b_recovery_3k.yaml`.

Hard rules of this notebook:

- **The teacher is NEVER rebuilt.** Stage C restores the existing `teacher_cache_v3` from Drive and fails loud on missing/fingerprint-mismatched cache.
- **Stage D verifies the donor is really text-only dense Qwen3.8-27B** before anything expensive (the 3.5-vs-3.8 mixup already burned one commit).
- **Stage P runs the 10-step `GRAD_CKPT=0` probe** and decides the production checkpointing mode automatically (headroom target: >=8 GiB, per the 5–8 GiB safety band).
- Run artifacts live in `run_<timestamp>/` with `checkpoints/ metrics/ config.json manifest.json`. **Never `rm -rf` an existing run dir**, local or Drive.
- Everything else — strict alpha=0, decimation=1, warm-step-0 protection, gated artifact selection — inherits strict-v3 unchanged.

Run every cell in order.

**Anti-burn safeguard:** never rely on the browser tab for a multi-hour run. Stage R can launch the trainer under nohup so closing Colab UI does not kill 3000 steps; progress is then watched via the DSH colab skill (poll_run.py) or by re-opening the notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# ---- Canonical Drive paths (v4) ----
import time
from pathlib import Path
import os, shutil, json, subprocess, sys, zipfile

DRIVE = Path('/content/drive/MyDrive')

def _first_existing(*candidates):
    for c in candidates:
        if Path(c).exists():
            return Path(c)
    return None

RELEASE_ZIP = _first_existing(
    DRIVE / 'mn_strict_kd_qwen38_27b_h100_rtxpro6000_v3.zip',
    *sorted(DRIVE.glob('mn_strict_kd_*27b*.zip')))
DONOR_DIR = _first_existing(
    DRIVE / 'colab_models' / 'Qwen3.8-27B',
    *(sorted((DRIVE / 'colab_models').glob('*Qwen3*27B*')) if (DRIVE/'colab_models').exists() else []))
WARM_STATE_ZIP = _first_existing(
    DRIVE / 'mn27' / 'state_max.zip',
    *sorted((DRIVE / 'mn27').glob('state*.zip'))) if (DRIVE / 'mn27').exists() else None
# Teacher cache + corpus carry LEGACY names from the 3.5/3.8 mislabel era.
# Content is validated later by fingerprints (cache_manifest vs donor index,
# corpus manifest); the name itself is not evidence either way - hence the
# loud warning rather than a hard fail on the new-style names only.
CACHE_DRIVE = _first_existing(
    DRIVE / 'mn27_strict_v3' / 'teacher_cache_v3')
if CACHE_DRIVE is None and (DRIVE / 'mn27_strict_v3').exists():
    hits = sorted((DRIVE / 'mn27_strict_v3').rglob('cache_manifest.json'))
    if hits: CACHE_DRIVE = hits[0].parent
DATA_DRIVE = _first_existing(
    DRIVE / 'mn27' / 'mix_qwen38_v3',
    DRIVE / 'mn27' / 'mix_qwen35_v3')  # legacy alias, fingerprint-checked below
if DATA_DRIVE is not None and DATA_DRIVE.name == 'mix_qwen35_v3':
    print('NOTE: corpus uses legacy 3.5-era folder name; its manifest will be '
          'validated against the Qwen3.8 donor index at cache-restore time.')

OUTPUT_DRIVE = Path('/content/drive/MyDrive/mn27_recovery_3k_v4')

WORK      = Path('/content/mn_recovery_3k')
PROJECT   = WORK / 'project'
RUN_NAME  = 'run_' + time.strftime('%Y%m%d_%H%M%S')
LOCAL_RUN = WORK / RUN_NAME          # checkpoints/ metrics/ manifests live here
TMP_CKPT  = WORK / 'ckpt_tmp'
(WORK / 'output').mkdir(parents=True, exist_ok=True)
LOCAL_RUN.mkdir(parents=True, exist_ok=True)
TMP_CKPT.mkdir(parents=True, exist_ok=True)
(LOCAL_RUN / 'metrics').mkdir(parents=True, exist_ok=True)
(LOCAL_RUN / 'checkpoints').mkdir(exist_ok=True)

# ---- one-shot status table: every gap visible in a single run ----
status = {
    'release_zip': RELEASE_ZIP,
    'donor_dir': DONOR_DIR,
    'warm_state_zip': WARM_STATE_ZIP,
    'teacher_cache': CACHE_DRIVE,
    'corpus': DATA_DRIVE,
}
missing = [k for k, v in status.items() if v is None]
for k, v in status.items():
    print(('[ OK ] ' if k not in missing else '[MISS] ') + k.ljust(14), v)
print()
assert RELEASE_ZIP is not None, (
    'Release zip NOT FOUND under MyDrive (searched mn_strict_kd_*27b*.zip). '
    'Upload it or fix the name.')
assert DONOR_DIR is not None, (
    'Donor dir NOT FOUND under colab_models/*Qwen3*27B*. '
    'Stage D will verify it is really text-only dense Qwen3.8.')
assert WARM_STATE_ZIP is not None, 'Warm state mn27/state_max.zip missing.'
assert CACHE_DRIVE is not None, (
    'Teacher cache NOT FOUND (mn27_strict_v3/teacher_cache_v3 or any '
    'cache_manifest.json under it). The teacher is never rebuilt in v4 - '
    'restore the Drive artifact first.')
assert DATA_DRIVE is not None, (
    'Corpus NOT FOUND (mix_qwen38_v3 or legacy mix_qwen35_v3). '
    'Corpus is restored, never rebuilt in v4.')

OUTPUT_DRIVE.mkdir(parents=True, exist_ok=True)
(OUTPUT_DRIVE / RUN_NAME / '_RUN_STARTED.txt').write_text(
    'v4 3k recovery started %s\nruns are append-only: never delete run dirs\n' % RUN_NAME)
print('Paths OK | RUN_NAME =', RUN_NAME)

In [ ]:
# Hardware + storage preflight
import torch, shutil, os
assert torch.cuda.is_available(), 'CUDA GPU required'
props = torch.cuda.get_device_properties(0)
GPU_NAME, GPU_GIB = props.name, props.total_memory / 2**30
print(f'GPU: {GPU_NAME} | {GPU_GIB:.2f} GiB | cc={torch.cuda.get_device_capability(0)} | bf16={torch.cuda.is_bf16_supported()}')
assert GPU_GIB >= 72, '27B recovery requires >=72 GiB VRAM'
assert torch.cuda.is_bf16_supported(), 'BF16 support required'
local_free = shutil.disk_usage('/content').free / 2**30
shm_free = shutil.disk_usage('/dev/shm').free / 2**30
print(f'free /content={local_free:.1f} GiB | /dev/shm={shm_free:.1f} GiB')
assert local_free >= 35, 'Need >=35 GiB free local disk even when donor stays on Drive'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:256'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [ ]:
# Extract release and install without touching Colab's CUDA-matched torch.
import shutil, zipfile, subprocess, sys, importlib
if PROJECT.exists(): shutil.rmtree(PROJECT)
tmp = WORK / 'extract_tmp'
if tmp.exists(): shutil.rmtree(tmp)
tmp.mkdir(parents=True)
with zipfile.ZipFile(RELEASE_ZIP) as zf: zf.extractall(tmp)
roots = [p for p in tmp.iterdir() if p.is_dir()]
src_root = roots[0] if len(roots) == 1 else tmp
shutil.move(str(src_root), str(PROJECT))
SRC = PROJECT / 'src'
assert (SRC / 'mn_strict_kd' / 'hardware.py').exists(), SRC
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
importlib.invalidate_caches()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT), '--no-deps'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT/'requirements-colab.txt')], check=True)
from mn_strict_kd.hardware import detect
profile = detect()
print(profile)
shutil.rmtree(WORK / 'extract_tmp', ignore_errors=True)

In [ ]:
# Ensure Transformers knows Qwen3.8, but never let pip replace torch/CUDA.
import importlib, subprocess, sys
ok = False
try:
    import transformers
    ok = hasattr(transformers, 'Qwen3_8ForCausalLM') and hasattr(transformers, 'Qwen3_8ForConditionalGeneration')
except Exception:
    pass
if not ok:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', '--no-deps',
                    'git+https://github.com/huggingface/transformers.git'], check=True)
    importlib.invalidate_caches()
    import transformers
print('transformers', transformers.__version__, 'Qwen3.8=', hasattr(transformers, 'Qwen3_8ForCausalLM'))
assert hasattr(transformers, 'Qwen3_8ForCausalLM')

In [ ]:
# ============================================================
# STAGE D -- DONOR VERIFICATION GATE (mandatory pre-launch #1)
# Fails loud unless the donor really is text-only dense Qwen3.8-27B.
# 3000 steps on a misidentified donor are worthless by definition.
# ============================================================
import subprocess, sys
checker = PROJECT / 'scripts' / 'check_donor_config.py'
if not checker.exists():
    # release zip predates the checker -> fetch it from the repo main branch
    import urllib.request
    url = 'https://raw.githubusercontent.com/kharkilirov1/memory-native/main/scripts/check_donor_config.py'
    checker.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, checker)
    print('fetched checker from main:', checker)
rc = subprocess.run([
    sys.executable, str(checker),
    '--donor', str(DONOR_DIR), '--min-total-gib', '46'
]).returncode
assert rc == 0, ('DONOR GATE FAILED rc=%d - do not launch on an unverified donor' % rc)
print('DONOR GATE PASSED')

In [ ]:
# Static release/warm-state validation.
preflight = LOCAL_RUN / 'metrics' / 'preflight_v3.json'
subprocess.run([
    sys.executable, str(PROJECT/'scripts/validate_strict_release_v3.py'),
    '--donor', str(DONOR_DIR), '--state', str(WARM_STATE_ZIP),
    '--output', str(preflight), '--require-gpu'
], check=True)
print(preflight.read_text())

In [ ]:
# GPU/Triton counter smoke on the ACTUAL card before any expensive 27B work.
subprocess.run([sys.executable, str(PROJECT/'scripts/gpu_counter_smoke_v3.py')], check=True)

In [ ]:
# Extract warm streamed state; normalize a possible nested zip root.
shm_free = shutil.disk_usage('/dev/shm').free / 2**30
base = Path('/dev/shm/mn27_state_v3') if shm_free >= 26 else WORK/'mn27_state_v3'
if base.exists(): shutil.rmtree(base)
base.mkdir(parents=True)
with zipfile.ZipFile(WARM_STATE_ZIP) as zf: zf.extractall(base)
manifests = [p for p in base.rglob('manifest.json') if list(p.parent.glob('block_*.pt'))]
assert len(manifests) == 1, f'Expected one streamed-state manifest, found {manifests}'
STATE_DIR = manifests[0].parent
print('STATE_DIR=', STATE_DIR)
print(json.dumps(json.loads((STATE_DIR/'manifest.json').read_text()), indent=2)[:3000])

In [ ]:
# Corpus: RESTORE ONLY (v4 hard rule). Missing corpus = stop, do not rebuild.
LOCAL_DATA = WORK / 'mix_qwen38_v3'
if LOCAL_DATA.exists(): shutil.rmtree(LOCAL_DATA)
LOCAL_DATA.mkdir(parents=True)
subprocess.run(['rsync','-a','--info=progress2',str(DATA_DRIVE)+'/',str(LOCAL_DATA)+'/'], check=True)
DATA_DIR = LOCAL_DATA
print((DATA_DIR/'manifest.json').read_text())

In [ ]:
# CPU unit tests for the corrected mathematics/selection/key mapping.
subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    str(PROJECT/'tests/test_strict_sparse_kd.py'),
    str(PROJECT/'tests/test_strict_gate_v3.py'),
    str(PROJECT/'tests/test_text_only_mapping.py')
], cwd=PROJECT, check=True)

## End-to-end 2-block smoke

Unchanged plumbing validation from v3 (teacher-cache schema, Qwen3.8 text-stack mapping,
warm-state remap, counter backward, strict loss and evaluation on 2 of 64 layers).
Plumbing only — not a quality measurement.

In [ ]:
# Build tiny 2-block cache + one strict student step.
SMOKE_CACHE = WORK/'smoke_cache'; SMOKE_OUT = WORK/'smoke_out'
for p in (SMOKE_CACHE, SMOKE_OUT):
    if p.exists(): shutil.rmtree(p)
    p.mkdir(parents=True)
env = os.environ.copy()
env.update({
    'MODEL': str(DONOR_DIR), 'DATA_DIR': str(DATA_DIR), 'OUT': str(SMOKE_CACHE),
    'STEPS':'1','BATCH':'1','SEQ':'128','TOPK':'128','SEED':'777',
    'DEVICE':'cuda','DTYPE':'bf16','NUM_BLOCKS':'2','CHUNK_ROWS':'1','SHARD_STEPS':'1','TEMPERATURE':'1.0'
})
subprocess.run([sys.executable, str(PROJECT/'scripts/kd_teacher_cache_v3.py')], env=env, check=True)
subprocess.run([sys.executable, str(PROJECT/'scripts/validate_kd_cache_v3.py'), str(SMOKE_CACHE),
                '--expected-topk','128','--expected-steps','1'], check=True)
env = os.environ.copy()
env.update({
    'MODEL': str(DONOR_DIR), 'STATE_DIR': str(STATE_DIR), 'DATA_DIR': str(DATA_DIR),
    'CACHE': str(SMOKE_CACHE), 'CKPT_DIR': str(SMOKE_OUT), 'DEVICE':'cuda',
    'STEPS':'1','NUM_BLOCKS':'2','KD_T':'1.0','KD_WEIGHT':'1.0','CE_WEIGHT':'1.0',
    'COUNTER_LR_START':'0.000125','COUNTER_LR_END':'0.000025','SCALE_LR':'0.000025',
    'DECIMATION':'1','FP_TRAIN_MODE':'none','GRAD_CKPT':'1','EVAL_EVERY':'1',
    'EVAL_MAX_TOKENS':'1024','EARLY_STOP_PATIENCE':'0','SAVE_BEST':'0',
    'MIN_IMPROVEMENT':'0.002','MAX_DOMAIN_REGRESSION':'0.05'
})
subprocess.run([sys.executable, str(PROJECT/'scripts/kd_cached_strict_v3.py')], env=env, check=True)
print((SMOKE_OUT/'run_summary.json').read_text())
shutil.rmtree(SMOKE_CACHE, ignore_errors=True); shutil.rmtree(SMOKE_OUT, ignore_errors=True)
torch.cuda.empty_cache()

## Stage C — Production teacher cache: RESTORE ONLY

`teacher_cache_v3`, K=1024, tail bucket, saved logsumexp. Fingerprint-validated against
the donor index and corpus manifest. There is **no rebuild path here on purpose**: if
validation fails, stop and resolve the mismatch explicitly — silently recomputing the
27B teacher pass is exactly the cost the 3k plan exists to avoid.

Do NOT change K=1024 before the first full 3k run; the compute bottleneck is student backward, not the cache.

In [ ]:
TOPK = profile.cache_topk
DONOR_INDEX = DONOR_DIR / 'model.safetensors.index.json'
def validate_cache(path):
    cmd = [sys.executable, str(PROJECT/'scripts/validate_kd_cache_v3.py'), str(path),
           '--expected-topk', str(TOPK), '--expected-steps', '400',
           '--model-index', str(DONOR_INDEX),
           '--data-manifest', str(DATA_DIR/'manifest.json')]
    return subprocess.run(cmd, check=False).returncode == 0

assert validate_cache(CACHE_DRIVE), (
    f'Teacher cache at {CACHE_DRIVE} FAILED validation - STOP. '
    'Rebuilding the teacher is forbidden in v4; resolve the mismatch explicitly. '
    '(expected-steps=400 pins the archived v3 build; a different value means the '
    'Drive copy changed since v3.)')
LOCAL_CACHE = WORK / 'teacher_cache_v3'
if LOCAL_CACHE.exists(): shutil.rmtree(LOCAL_CACHE)
LOCAL_CACHE.mkdir(parents=True)
subprocess.run(['rsync','-a','--info=progress2',str(CACHE_DRIVE)+'/',str(LOCAL_CACHE)+'/'], check=True)
CACHE = LOCAL_CACHE
print(json.dumps(json.loads((CACHE/'cache_manifest.json').read_text()), indent=2))

## Stage P — 10-step `GRAD_CKPT=0` VRAM probe (mandatory pre-launch #2)

Production shapes (micro_batch 2 x seq 512), 10 steps, while polling real GPU memory.
Decision rule: **peak + 8 GiB <= total => GRAD_CKPT=0 for all 3000 steps, else GRAD_CKPT=1.**
Selective per-block checkpointing needs a runner-side patch (`production/V4_CHANGES.md`)
and is deliberately not attempted here. No experiments mid-run after this decision.

In [ ]:
import threading
def peak_gpu_gib(poll_s=2.0):
    stop, peak = threading.Event(), [0.0]
    def loop():
        import subprocess as sp
        while not stop.is_set():
            try:
                out = sp.run(['nvidia-smi','--query-gpu=memory.used','--format=csv,noheader,nounits'],
                             capture_output=True, text=True, timeout=10).stdout.strip().splitlines()
                peak[0] = max(peak[0], max(float(x) for x in out if x.strip()))
            except Exception:
                pass
            stop.wait(poll_s)
    t = threading.Thread(target=loop, daemon=True); t.start()
    return peak, stop

def run_probe(grad_ckpt):
    out_dir = WORK / ('probe_gck%d' % grad_ckpt)
    if out_dir.exists(): shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)
    env = os.environ.copy()
    env.update({
        'MODEL': str(DONOR_DIR), 'STATE_DIR': str(STATE_DIR), 'DATA_DIR': str(DATA_DIR),
        'CACHE': str(CACHE), 'CKPT_DIR': str(out_dir), 'CKPT_TMP': str(TMP_CKPT), 'DEVICE':'cuda',
        'STEPS':'10','NUM_BLOCKS':'0','BATCH':'2','SEQ':'512',
        'KD_T':'1.0','KD_WEIGHT':'1.0','CE_WEIGHT':'1.0',
        'COUNTER_LR_START':'0.000125','COUNTER_LR_END':'0.00001','SCALE_LR_START':'0.000025',
        'SCALE_LR_END':'0.000005','FP_LR':'0.0','GRAD_CLIP':'0.5',
        'STATS_SCOPE':'group','DECIMATION':'1','EVAL_EVERY':'99999','EVAL_MAX_TOKENS':'4096',
        'LOG_EVERY':'1','GRAD_CKPT':str(grad_ckpt),'FP_TRAIN_MODE':'none',
        'SAVE_BEST':'0','EARLY_STOP_PATIENCE':'0'
    })
    log = out_dir / 'probe.log'
    peak, stop = peak_gpu_gib()
    rc = None
    try:
        with log.open('w', buffering=1) as lf:
            pr = subprocess.Popen([sys.executable, str(PROJECT/'scripts/kd_cached_strict_v3.py')],
                                  env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                  text=True, bufsize=1)
            for line in pr.stdout:
                print(line, end=''); lf.write(line)
            rc = pr.wait()
    finally:
        time.sleep(2); stop.set()
    used = peak[0] / 1024.0
    print('[GRAD_CKPT=%d] rc=%s peak_used=%.2f GiB | headroom=%.2f GiB' % (grad_ckpt, rc, used, GPU_GIB - used))
    return rc, used

torch.cuda.empty_cache()
rc0, used0 = run_probe(0)
if rc0 == 0 and (GPU_GIB - used0) >= 8.0:
    PROD_GRAD_CKPT = 0
    reason = 'GRAD_CKPT=0 fits with >=8 GiB headroom'
else:
    if rc0 != 0:
        reason = 'GRAD_CKPT=0 did not complete -> fallback to 1'
    else:
        reason = 'headroom %.2f GiB < 8 GiB target -> conservative fallback to 1' % (GPU_GIB - used0)
        print(reason)
    torch.cuda.empty_cache()
    rc1, used1 = run_probe(1)
    assert rc1 == 0, 'GRAD_CKPT=1 probe must succeed - no working mode, cannot proceed'
    PROD_GRAD_CKPT = 1
decision = {'gpu': GPU_NAME, 'total_gib': round(GPU_GIB, 2),
            'probe_gck0_rc': rc0, 'probe_gck0_peak_gib': round(used0, 2),
            'decision_grad_ckpt': PROD_GRAD_CKPT, 'reason': reason}
(LOCAL_RUN / 'metrics' / 'gck_decision.json').write_text(json.dumps(decision, indent=2))
print(json.dumps(decision, indent=2))
torch.cuda.empty_cache()

## Production strict KD — 3000 steps

Policy fixed for this run (see `production/qwen38_27b_recovery_3k.yaml`):

- full eval every 500 steps at the warm-comparison protocol tokens (`EVAL_EVERY=500`);
- early-stop patience 3 non-improving FULL evals, min improvement 0.001;
- LR `counter 1.25e-4 -> 1.0e-5`, `scale 2.5e-5 -> 5e-6`;
- **no ordinary gradient accumulation** (strict doctrine): micro-batch stays 2;
- periodic mid-run full checkpoints and cheap proxy evals are runner-side patches
  (`V4_CHANGES.md`) — with the unpatched runner the accepted/deployable artifact is still
  gate-authoritative `best.pt`, protected additionally by warm-state rollback.

Warm step 0 stays authoritative unless a KD eval passes every gate. A falling KD loss alone is **not** evidence of improvement.

In [ ]:
# Resolve and record the exact production recipe.
recipe = json.loads((PROJECT/'configs/qwen38_27b_strict_kd_v3.json').read_text())
recipe['hardware'] = profile.__dict__
recipe['paths'] = {'donor': str(DONOR_DIR), 'warm_state_dir': str(STATE_DIR),
                   'data': str(DATA_DIR), 'cache': str(CACHE), 'output_run': str(LOCAL_RUN)}
recipe['grad_checkpointing_decision'] = json.loads((LOCAL_RUN/'metrics'/'gck_decision.json').read_text())
(LOCAL_RUN/'config.json').write_text(json.dumps(recipe, indent=2))
print(json.dumps(recipe, indent=2)[:4000])

## Stage R — anti-burn launcher (recommended for the real 3k)

A closed tab kills the kernel and 3000 steps with it. This stage writes an env-complete `dsh_launch_v4.sh` so the trainer runs under `nohup`, surviving UI closure. Afterwards watch it with the DSH colab skill (`poll_run.py`) or `tail -f` in a re-opened session. Run it AFTER the production-env cell; then run `bash .../dsh_launch_v4.sh` INSTEAD of the interactive launch cell.

In [ ]:
# Stage R -- anti-burn launcher (recommended for the real 3k).
# The interactive launch cell below works too; if you plan to CLOSE the browser
# tab mid-run, use this instead: it writes dsh_launch_v4.sh embedding the exact
# production env and starts the trainer under nohup, immune to UI closure.
launcher = WORK / 'dsh_launch_v4.sh'
env = env  # noqa: F821 - production env dict from cell a0020 runs first
if 'env' not in dir():
    raise RuntimeError('Stage R must run AFTER the production-env cell (a0020)')
env_lines = ''.join(
    "export %s=%s\n" % (k, ("'%s'" % v) if (' ' in str(v) or "'" in str(v)) else str(v))
    for k, v in sorted(env.items())
)
script = (
    "#!/usr/bin/env bash\n"
    "set -euo pipefail\n"
    "cd " + str(WORK) + "\n"
    "RUN=" + RUN_NAME + "\n"
    'mkdir -p "$RUN/nohup"\n'
    "if pgrep -f kd_cached_strict_v3.py >/dev/null; then echo already running; exit 1;\n"
    "fi\n" + env_lines +
    'nohup python project/scripts/kd_cached_strict_v3.py'
    ' > "$RUN/strict_v4_console.log" 2>&1 < /dev/null &\n'
    "echo launched pid $!\n"
)
launcher.write_text(script)
print('wrote', launcher)
print('Launch detached from a Colab terminal:')
print('  bash /content/mn_recovery_3k/dsh_launch_v4.sh')
print('Then close the browser freely; monitor via poll_run.py or a later notebook session.')


In [ ]:
# Run strict KD v4/3k.
for name in ['metrics.json','selection.json','best.pt','selected_artifact.json',
             'run_summary.json','KD_ACCEPTED.txt','USE_WARM_STATE.txt','restore_meta.json']:
    p = LOCAL_RUN / name
    if p.exists(): p.unlink()

env = os.environ.copy(); env.update({
    'MODEL': str(DONOR_DIR), 'STATE_DIR': str(STATE_DIR), 'DATA_DIR': str(DATA_DIR),
    'CACHE': str(CACHE), 'CKPT_DIR': str(LOCAL_RUN), 'CKPT_TMP': str(TMP_CKPT),
    'DEVICE': 'cuda',
    'STEPS': '3000', 'BATCH': '2', 'SEQ': '512',
    'KD_T': '1.0', 'KD_WEIGHT': '1.0', 'CE_WEIGHT': '1.0',
    'COUNTER_LR_START': '0.000125', 'COUNTER_LR_END': '0.00001',
    'SCALE_LR_START': '0.000025', 'SCALE_LR_END': '0.000005',
    'FP_LR': '0.0', 'GRAD_CLIP': '0.5', 'STATS_SCOPE': 'group', 'DECIMATION': '1',
    'EVAL_EVERY': '500', 'EVAL_MAX_TOKENS': '24000', 'NUM_BLOCKS': '0', 'LOG_EVERY': '10',
    'GRAD_CKPT': str(PROD_GRAD_CKPT), 'FP_TRAIN_MODE': 'none', 'SAVE_BEST': '1',
    'EARLY_STOP_PATIENCE': '3', 'MIN_IMPROVEMENT': '0.001',
    'MAX_DOMAIN_REGRESSION': '0.05'
})
logfile = LOCAL_RUN / 'strict_v4_console.log'
with logfile.open('w', buffering=1) as log:
    proc = subprocess.Popen([sys.executable, str(PROJECT/'scripts/kd_cached_strict_v3.py')],
                            env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end=''); log.write(line)
    rc = proc.wait()
if rc != 0:
    print(logfile.read_text(errors='replace')[-16000:])
    raise RuntimeError(f'strict v4 failed with rc={rc}')
print('RUN COMPLETE:', RUN_NAME)

In [ ]:
# Authoritative selection (unchanged gates).
selection = json.loads((LOCAL_RUN/'selection.json').read_text())
print(json.dumps(selection, indent=2, ensure_ascii=False))
if selection['accepted_kd']:
    assert (LOCAL_RUN/'best.pt').exists(), 'Gate accepted KD but deployable best.pt is missing'
else:
    assert (LOCAL_RUN/'USE_WARM_STATE.txt').exists(), 'rejected KD must keep warm-state marker'
print((LOCAL_RUN/'run_summary.json').read_text())

In [ ]:
# Provenance manifest + sync ONLY gate-approved artifacts. Runs are append-only.
import hashlib
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    'run_name': RUN_NAME,
    'gpu': GPU_NAME, 'gpu_gib': round(GPU_GIB, 2),
    'donor_dir': str(DONOR_DIR),
    'donor_config_sha256': sha256(DONOR_DIR/'config.json'),
    'donor_index_sha256': sha256(DONOR_INDEX),
    'release_zip': str(RELEASE_ZIP),
    'teacher_cache_drive': str(CACHE_DRIVE),
    'warm_state_zip': str(WARM_STATE_ZIP),
    'data_drive': str(DATA_DRIVE),
    'grad_checkpointing': PROD_GRAD_CKPT,
    'finished_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
(LOCAL_RUN/'manifest.json').write_text(json.dumps(manifest, indent=2))

if selection['accepted_kd']:
    best_local = LOCAL_RUN / 'best.pt'
    shutil.move(str(best_local), str(LOCAL_RUN/'checkpoints'/'best.pt'))

small = ['preflight_v3.json','resolved_recipe_v3.json','config.json','manifest.json',
         'gck_decision.json','metrics.json','selection.json','selected_artifact.json',
         'run_summary.json','strict_v4_console.log','restore_meta.json',
         'KD_ACCEPTED.txt','USE_WARM_STATE.txt']
dst = OUTPUT_DRIVE / RUN_NAME
dst.mkdir(parents=True, exist_ok=True)
for name in small:
    p = LOCAL_RUN / name
    if p.exists():
        shutil.copy2(p, dst / name)
if selection['accepted_kd']:
    subprocess.run(['rsync','-a','--info=progress2',
                    str(LOCAL_RUN/'checkpoints'/'best.pt'), str(dst/'checkpoints'/'best.pt')], check=True)
(OUTPUT_DRIVE / RUN_NAME / '_COMPLETE').write_text(
    'v4 3k recovery finished; selection.json is authoritative\n')
print('Synced:', dst)
print('Reminder: runs are append-only - never delete existing run dirs.')

## Interpretation & early stopping guidance

- `KD_ACCEPTED.txt` + `best.pt`: strict-alpha=0 candidate beat warm aggregate metric within margin AND stayed inside the 5% per-domain regression cap.
- `USE_WARM_STATE.txt`: KD did not earn deployment; keep `/MyDrive/mn27/state_max.zip`.
- `selection.json` is authoritative; a falling KD loss alone is NOT evidence.

Early stopping — 3000 is an upper bound:

```text
plateau pattern (example)      keep-going pattern (example)
500   1.91                     500   1.94
1000  1.84                     1000  1.90
1500  1.80                     1500  1.87
2000  1.79   <- stop ~2000     2000  1.84
2500  1.789                    2500  1.82   -> ride to 3000
3000  1.789
```

Reference anchors: v3 observed aggregate `2.024199 -> 1.938789` over its short window.
For a healthy 3k recovery, ~**1.85–1.92** would be a strong outcome; approaching the FP
baseline would be very interesting for 27B ternary recovery. Do not extrapolate the first
225 steps linearly.